# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [26]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [27]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [28]:
# TODO
print(df.shape)
print()
print(df.dtypes)
print()
print(df.isnull().sum())
print()
print(df.duplicated().sum())

(8, 6)

order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

1


**What is wrong with this data?** List at least five specific problems:

1. price as an object
2. duplicated rows
3. nulls in some entries
4. ts (time) as an object
5. inconsistent spelling of items

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [29]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean =    df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)
log('duplicates', 'dropped exact duplicate rows', removed)


[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [30]:
# TODO: clean['price'] = ...
clean['price'] = clean['price'].astype(str).str.strip().str.replace('$', '', regex=False).astype(float)

assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text
log('price', 'stripped $ and whitespace, cast to float — arrived as text due to mixed $/plain formatting', clean['price'].tolist())

[price] stripped $ and whitespace, cast to float — arrived as text due to mixed $/plain formatting ([7.5, 7.5, 12.0, 24.0, 6.0, 6.0, 12.0] row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [31]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty']<0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately
missing_qty = clean[clean['qty'].isna()].copy()
clean = clean[clean['qty'].notna()].copy()
log('qty_missing', 'dropped rows with unrecoverable missing qty (kept aside, not deleted)', missing)

negative_qty = clean[clean['qty']<0].copy()
clean = clean[clean['qty']>=0].copy()
log('qty_negative', 'dropped rows with negative qty (kept aside, not deleted)', negative)

clean['qty'] = clean['qty'].astype(int)


[qty_missing] dropped rows with unrecoverable missing qty (kept aside, not deleted) (1 row(s))
[qty_negative] dropped rows with negative qty (kept aside, not deleted) (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [32]:
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z0-9]', '', regex=True))
# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Apparel',
    'raingear': 'Rain Gear',
}
clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))
# print('after: ', sorted(clean['category'].unique()))
before_n = df['category'].nunique()
after_n = clean['category'].nunique()
log('category_normalize', 'lowercased/stripped punctuation, mapped variants to canonical labels', f'{before_n} -> {after_n}')

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'Rain Gear']
[category_normalize] lowercased/stripped punctuation, mapped variants to canonical labels (6 -> 4 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [33]:
# TODO
print('before:', sorted(clean['item'].dropna().unique()))

# normalize whitespace/case for comparison, but keep a display-friendly canonical form
clean['item'] = clean['item'].astype(str).str.strip()
clean['item'] = clean['item'].replace('nan', np.nan)  # str(NaN) -> 'nan', restore true NaN

# same product, two spellings/cases -> map explicitly rather than trust fuzzy matching
ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
}
mask = clean['item'].notna()
clean.loc[mask, 'item'] = clean.loc[mask, 'item'].str.lower().map(ITEM_MAP).fillna(clean.loc[mask, 'item'])

print('after: ', sorted(clean['item'].dropna().unique()))

# missing item name (order_id 7): same reasoning as missing qty — nothing in the row
# (category=Merch, price=12) narrows it down to one specific product, so guessing
# would fabricate a name. Drop from `clean`, keep aside for the source-data owner.
missing_item = clean[clean['item'].isna()].copy()
clean = clean[clean['item'].notna()].copy()

log('item_normalize', 'mapped Cheeseburger/cheese burger to one canonical spelling', clean['item'].nunique())
log('item_missing', 'dropped row with no item name (kept aside, not deleted)', len(missing_item))

before: ['Cheeseburger', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'UVA T-Shirt', 'rain poncho']
[item_normalize] mapped Cheeseburger/cheese burger to one canonical spelling (3 row(s))
[item_missing] dropped row with no item name (kept aside, not deleted) (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [34]:
# TODO
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed = clean['ts'].isna().sum()
log('ts_parse', 'parsed ts to datetime, coerced unparseable/empty values to NaT', failed)

clean['hour'] = clean['ts'].dt.hour
clean[['ts', 'hour']]



[ts_parse] parsed ts to datetime, coerced unparseable/empty values to NaT (3 row(s))


,ts,hour
0,2026-09-05 12:03:00,12.0
2,NaT,NaN
4,NaT,NaN
6,NaT,NaN


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [35]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
clean = clean[clean['qty'].notna()].copy()
clean['is_refund'] = clean['qty'] < 0
clean['qty'] = clean['qty'].astype(int)

In [36]:
# TODO: assertions
# 1. no exact duplicate rows survived Step 1
assert clean.duplicated().sum() == 0, "duplicate rows leaked through"

# 2. price is a real float, no $ or text sneaked back in
assert clean['price'].dtype == float, "price is not float"
assert (clean['price'] > 0).all(), "found a non-positive price"

# 3. qty is a real integer, and nothing null survived (Step 3's drop should be complete)
assert clean['qty'].dtype == int, "qty is not int"
assert clean['qty'].notna().all(), "null qty leaked through"
assert (clean['qty'] != 0).all(), "found a zero qty row (should be dropped or meaningful, never 0)"

# 4. is_refund flag is consistent with the sign of qty (Step 3's flagging logic)
assert (clean['is_refund'] == (clean['qty'] < 0)).all(), "is_refund out of sync with qty sign"

# 5. category only contains the canonical labels from Step 4 -- no stray spellings
assert clean['category'].isin(['Food', 'Merch', 'Apparel', 'Rain Gear']).all(), "uncanonicalized category found"

# 6. item is never null (Step 5's drop should be complete) and only canonical spellings remain
assert clean['item'].notna().all(), "null item leaked through"
assert 'cheese burger' not in clean['item'].str.lower().values, "uncollapsed item spelling found"

# 7. ts is real datetime64, and any NaT is accounted for (Step 6)
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), "ts is not datetime64"

print(f"all {7} assertion blocks passed")

# TODO: clean['revenue'] = ...
clean['revenue'] = clean['price'] * clean['qty']

# TODO: print rows, units, revenue, distinct categories
print('rows:              ', len(clean))
print('units (net):       ', clean['qty'].sum())
print('revenue (net):      $%.2f' % clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique(), sorted(clean['category'].unique()))

all 7 assertion blocks passed
rows:               4
units (net):        9
revenue (net):      $94.50
distinct categories: 3 ['Apparel', 'Food', 'Rain Gear']


In [40]:
clean

,order_id,item,category,qty,price,ts,hour,is_refund,revenue
0,1,Cheeseburger,Food,2,7.5,2026-09-05 12:03:00,12.0,False,15.0
2,2,Cheeseburger,Food,1,7.5,NaT,NaN,False,7.5
4,4,UVA T-Shirt,Apparel,2,24.0,NaT,NaN,False,48.0
6,6,rain poncho,Rain Gear,4,6.0,NaT,NaN,False,24.0


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [37]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,"stripped $ and whitespace, cast to float — arr...","[7.5, 7.5, 12.0, 24.0, 6.0, 6.0, 12.0]"
2,qty_missing,dropped rows with unrecoverable missing qty (k...,1
3,qty_negative,"dropped rows with negative qty (kept aside, no...",1
4,category_normalize,"lowercased/stripped punctuation, mapped varian...",6 -> 4
5,item_normalize,mapped Cheeseburger/cheese burger to one canon...,3
6,item_missing,"dropped row with no item name (kept aside, not...",1
7,ts_parse,"parsed ts to datetime, coerced unparseable/emp...",3


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [42]:
# Checkpoint
rows_after = 4            # TODO
revenue_after = '$94.50'         # TODO
biggest_decision = 'Poncho as a refund, rather than a positive sale'    # TODO: which choice moved the number most
revenue_other_way = '$112.5'     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 4
revenue: $94.50
decision that mattered: Poncho as a refund, rather than a positive sale
revenue the other way: $112.5
